##  Back propagation using a computation graph

Working through this, we will gain insight into a key algorithm used by most machine learning frameworks. Gradient descent requires the derivative of the cost with respect to each parameter in the network. Neural networks can have millions or even billions of parameters. The back propagation algorithm is used to compute those derivatives. Computation graphs are used to simplify the operation.

In [2]:
# Importing the dependencies
from sympy import *
import numpy as np
import re
%matplotlib widget
import matplotlib.pyplot as plt

## Computation Graph

A computation graph simplifies the computation of complex derivatives by breaking them into smaller steps.

We will be calculating the derivative of this slightly complex expression, 𝐽=(2+3𝑤)2. We would be finding the derivative of 𝐽 with respect to 𝑤 or ∂𝐽∂𝑤


## Forward Propagation

In [3]:
# Calculating the values in the forward direction.
w = 3
a = 2+3*w
J = a**2
print(f"a = {a}, J = {J}")

a = 11, J = 121


## Backprop

Backprop is the algorithm we use to calculate derivatives. Backprop starts at the right and moves to the left. The first node to consider is  𝐽=𝑎2 and the first step is to find  ∂𝐽∂𝑎

∂𝐽∂𝑎
 
Arithmetically finding ∂𝐽∂𝑎 by finding how 𝐽 changes as a result of a little change in 𝑎.

In [4]:
a_epsilon = a + 0.001       # a epsilon
J_epsilon = a_epsilon**2    # J_epsilon
k = (J_epsilon - J)/0.001   # difference divided by epsilon
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_da ~= k = {k} ")

J = 121, J_epsilon = 121.02200099999999, dJ_da ~= k = 22.000999999988835 


∂𝐽∂𝑎 is 22 which is 2×𝑎. Our result is not exactly 2×𝑎 because our epsilon value is not infinitesimally small.

Symbolically, let's use SymPy to calculate derivatives symbolically as we did in the derivatives. We will prefix the name of the variable with an 's' to indicate this is a symbolic variable.

In [5]:
sw,sJ,sa = symbols('w,J,a')
sJ = sa**2
sJ

a**2

In [6]:
sJ.subs([(sa,a)])

121

In [7]:
dJ_da = diff(sJ, sa)
dJ_da

2*a

So, ∂𝐽∂𝑎=2𝑎, When  𝑎=11, ∂𝐽∂𝑎=22. This matches our arithmetic calculation above.

∂𝐽∂𝑤

Moving from right to left, the next value we would like to compute is ∂𝐽∂𝑤. To do this, we first need to calculate ∂𝑎∂𝑤 which describes how the output of this node, 𝑎, changes when the input 𝑤 changes a little bit.

Arithmetically
Finding  ∂𝑎∂𝑤 by finding how  𝑎 changes as a result of a little change in  𝑤.

In [8]:
w_epsilon = w + 0.001       # a  plus a small value, epsilon
a_epsilon = 2 + 3*w_epsilon
k = (a_epsilon - a)/0.001   # difference divided by epsilon
print(f"a = {a}, a_epsilon = {a_epsilon}, da_dw ~= k = {k} ")

a = 11, a_epsilon = 11.003, da_dw ~= k = 3.0000000000001137 


Calculated arithmetically,  ∂𝑎∂𝑤≈3

In [9]:
sa = 2 + 3*sw
sa

3*w + 2

In [10]:
da_dw = diff(sa,sw)
da_dw

3

We know that a small change in  𝑤 will cause  𝑎 to change by 3 times that amount.
We know that a small change in  𝑎will cause  𝐽 to change by  2×𝑎 times that amount. (a=11 in this example)
so, putting these together, We know that a small change in  𝑤 will cause  𝐽 to change by  3×2×𝑎 times that amount.
These cascading changes go by the name of the chain rule.

In [11]:
dJ_dw = da_dw * dJ_da
dJ_dw

6*a

And  𝑎 is 11 in this example so  ∂𝐽∂𝑤=66. We can check this arithmetically

In [12]:
w_epsilon = w + 0.001
a_epsilon = 2 + 3*w_epsilon
J_epsilon = a_epsilon**2
k = (J_epsilon - J)/0.001   # difference divided by epsilon
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_dw ~= k = {k} ")

J = 121, J_epsilon = 121.06600900000001, dJ_dw ~= k = 66.0090000000082 


Another view
We could visualize these cascading changes this way:
A small change in  𝑤 is multiplied by  ∂𝑎∂𝑤 resulting in a change that is 3 times as large. This larger change is then multiplied by  ∂𝐽∂𝑎 resulting in a change that is now  3×22=66 times larger.

## Computation Graph of a Simple Neural Network¶

Forward propagation

In [13]:
# Inputs and parameters
x = 2
w = -2
b = 8
y = 1
# calculate per step values   
c = w * x
a = c + b
d = a - y
J = d**2/2
print(f"J={J}, d={d}, a={a}, c={c}")

J=4.5, d=3, a=4, c=-4


Backward propagation (Backprop)

backprop starts at the right and moves to the left. The first node to consider is  𝐽=12𝑑2 and the first step is to find  ∂𝐽∂𝑑

∂𝐽∂𝑑

Arithmetically Find  ∂𝐽∂𝑑 by finding how  𝐽 changes as a result of a little change in  𝑑.

In [14]:
d_epsilon = d + 0.001
J_epsilon = d_epsilon**2/2
k = (J_epsilon - J)/0.001   # difference divided by epsilon
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_dd ~= k = {k} ")

J = 4.5, J_epsilon = 4.5030005, dJ_dd ~= k = 3.0004999999997395 


∂𝐽∂𝑑 is 3, which is the value of 𝑑. Our result is not exactly 𝑑 because our epsilon value is not infinitesimally small.

Symbolically, let's use SymPy to calculate derivatives symbolically

In [15]:
sx,sw,sb,sy,sJ = symbols('x,w,b,y,J')
sa, sc, sd = symbols('a,c,d')
sJ = sd**2/2
sJ

d**2/2

In [16]:
sJ.subs([(sd,d)])

9/2

In [17]:
dJ_dd = diff(sJ, sd)
dJ_dd

d

So,  ∂𝐽∂𝑑 = d. When  𝑑=3,  ∂𝐽∂𝑑 = 3. This matches our arithmetic calculation above.

∂𝐽∂𝑎
 
Moving from right to left, the next value we would like to compute is ∂𝐽∂𝑎. To do this, we first need to calculate ∂𝑑∂𝑎 which describes how the output of this node changes when the input 𝑎 changes a little bit. (we are not interested in how the output changes when 𝑦 changes since 𝑦 is not a parameter.)

Arithmetically,
Finding  ∂𝑑∂𝑎 by finding how  𝑑 changes as a result of a little change in  𝑎.

In [18]:
a_epsilon = a + 0.001         # a  plus a small value
d_epsilon = a_epsilon - y
k = (d_epsilon - d)/0.001   # difference divided by epsilon
print(f"d = {d}, d_epsilon = {d_epsilon}, dd_da ~= k = {k} ")

d = 3, d_epsilon = 3.0010000000000003, dd_da ~= k = 1.000000000000334 


Calculated arithmetically,  ∂𝑑∂𝑎≈1. Let's try it with SymPy.

Symbolically

In [19]:
sd = sa - sy
sd

a - y

In [20]:
dd_da = diff(sd,sa)
dd_da

1

Calculated arithmetically,  ∂𝑑∂𝑎 also equals 1.

We know that a small change in  𝑎 will cause  𝑑 to change by 1 times that amount.
We know that a small change in  𝑑 will cause  𝐽 to change by  𝑑 times that amount. (d=3 in this example)
so, putting these together, We know that a small change in  𝑎 will cause  𝐽 to change by  1×𝑑 times that amount.
This is again the chain rule

In [21]:
dJ_da = dd_da * dJ_dd
dJ_da

d

And  𝑑 is 3 in this example so  ∂𝐽∂𝑎=3. We can check this arithmetically

In [22]:
a_epsilon = a + 0.001
d_epsilon = a_epsilon - y
J_epsilon = d_epsilon**2/2
k = (J_epsilon - J)/0.001   
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_da ~= k = {k} ")

J = 4.5, J_epsilon = 4.503000500000001, dJ_da ~= k = 3.0005000000006277 


they match! 

The steps in Backprop: we can write down the basic method: working right to left, for each node:

calculate the local derivative(s) of the node using the chain rule, combine with the derivative of the cost with respect to the node to the right. The 'local derivative(s)' are the derivative(s) of the output of the current node with respect to all inputs or parameters.

∂𝐽∂𝑐, ∂𝐽∂𝑏

The next node has two derivatives of interest. We need to calculate ∂𝐽∂𝑐 so we can propagate to the left. We also want to calculate ∂𝐽∂𝑏. Finding the derivative of the cost with respect to the parameters 𝑤 and 𝑏 is the object of backprop. We will find the local derivatives, ∂𝑎∂𝑐 and ∂𝑎∂𝑏 first and then combine those with the derivative coming from the right, ∂𝐽∂𝑎.

In [23]:
# calculate the local derivatives da_dc, da_db
sa = sc + sb
sa

b + c

In [24]:
da_dc = diff(sa,sc)
da_db = diff(sa,sb)
print(da_dc, da_db)

1 1


In [25]:
dJ_dc = da_dc * dJ_da
dJ_db = da_db * dJ_da
print(f"dJ_dc = {dJ_dc},  dJ_db = {dJ_db}")

dJ_dc = d,  dJ_db = d


And in our example, d = 3

∂𝐽∂𝑤
 
The last node in this example calculates c. Here, we are interested in how J changes with respect to the parameter w. We will not back propagate to the input 𝑥, so we are not interested in ∂𝐽∂𝑥. Let's start by calculating ∂𝑐∂𝑤.

In [26]:
# calculate the local derivative
sc = sw * sx
sc

w*x

In [27]:
dc_dw = diff(sc,sw)
dc_dw

x

This derivative will vary depending on the value of  𝑥. This is 2 in our example.

Combine this with  ∂𝐽∂𝑐 to find  ∂𝐽∂𝑤.

In [28]:
dJ_dw = dc_dw * dJ_dc
dJ_dw

d*x

In [29]:
print(f"dJ_dw = {dJ_dw.subs([(sd,d),(sx,x)])}")

dJ_dw = 2*d


𝑑=3, so  ∂𝐽∂𝑤=6 for our example.
Let's test this arithmetically

In [30]:
J_epsilon = ((w+0.001)*x+b - y)**2/2
k = (J_epsilon - J)/0.001  
print(f"J = {J}, J_epsilon = {J_epsilon}, dJ_dw ~= k = {k} ")

J = 4.5, J_epsilon = 4.506002, dJ_dw ~= k = 6.001999999999619 


they match!

## Final Outcome

worked through an example of back propagation using a computation graph. We can apply this to larger examples by following the same node by node approach.